# Experiment 1 — headline benchmark (PD)

5-fold CV, NO_HPO + HPO, all 14 PD datasets. Default-probability prediction: discrimination (AUC/Gini/KS), calibration (Brier/ECE), imbalance-aware metrics (AP_normalized), tuning effect, and cost.

Figures → `figures/experiment1/pd/` (wiped on rerun).

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.paths import results_root
from src.visualizations.experiment_plots import (
    apply_style, reset_figure_dir, load_summary,
    performance_heatmap, method_ranking_bars, per_dataset_bars,
    learning_curve, imbalance_curve, metric_boxplots,
    hpo_improvement_bars, runtime_performance_scatter,
)
apply_style()

RESULTS_ROOT = results_root()          # override here if your copy lives elsewhere
SUMMARY_DIR  = RESULTS_ROOT / 'summaries'
FIGURES_DIR  = reset_figure_dir(PROJECT_ROOT / 'figures' / 'experiment1/pd')
print('results:', RESULTS_ROOT, '| figures:', FIGURES_DIR)

In [ ]:
df      = load_summary(SUMMARY_DIR, experiment='experiment1', task='pd')          # NO_HPO
df_both = load_summary(SUMMARY_DIR, experiment='experiment1', task='pd', hpo_mode=None)
print(f'{df["method"].nunique()} methods x {df["dataset"].nunique()} datasets')

## Discrimination

In [ ]:
performance_heatmap(df, 'AUC', task_name='PD', out_dir=FIGURES_DIR)
method_ranking_bars(df, 'AUC', task_name='PD', out_dir=FIGURES_DIR)
metric_boxplots(df, 'AUC', task_name='PD', out_dir=FIGURES_DIR)

In [ ]:
performance_heatmap(df, 'Gini', task_name='PD', out_dir=FIGURES_DIR)
performance_heatmap(df, 'KS',   task_name='PD', out_dir=FIGURES_DIR)

## Imbalance-aware ranking quality (prevalence-corrected AP)

In [ ]:
performance_heatmap(df, 'AP_normalized', task_name='PD', out_dir=FIGURES_DIR)
method_ranking_bars(df, 'AP_normalized', task_name='PD', out_dir=FIGURES_DIR)

## Calibration (lower is better)

In [ ]:
performance_heatmap(df, 'Brier', task_name='PD', cmap='RdYlGn_r', out_dir=FIGURES_DIR)
performance_heatmap(df, 'ECE',   task_name='PD', cmap='RdYlGn_r', out_dir=FIGURES_DIR)

## Who wins where, and by how much

In [ ]:
from src.utils.statistical_testing import metric_matrix, plot_win_loss_matrix, plot_pama_bars, wlt_summary, pama
mat = metric_matrix(df, 'AUC')
plot_win_loss_matrix(mat, out_path=FIGURES_DIR / 'pd_win_loss_auc.pdf')
plot_pama_bars(mat, metric_name='AUC', out_path=FIGURES_DIR / 'pd_pama_auc.pdf')
display(wlt_summary(mat)); display(pama(mat))

## Does hyper-parameter tuning help? (foundation models are copies → 0 by design)

In [ ]:
hpo_improvement_bars(df_both, 'AUC', task_name='PD', out_dir=FIGURES_DIR)

## Cost / quality frontier

In [ ]:
runtime_performance_scatter(df, 'AUC', task_name='PD', out_dir=FIGURES_DIR)